# 14. Polars — `polars`
**Python Libraries Lab · Data Science**

_A very fast, modern DataFrame library with a clean expression API._

Polars does the same job as pandas, tables of data, but it is written in Rust and uses every core of your machine. On big files it is often many times faster. Its style is a little different. You describe what you want with expressions, like p l dot col revenue dot sum, and Polars works out the fastest way to do it.

**In this notebook:** You will repeat the pandas analysis of ShopEasy orders in Polars, revenue by city and by month, then write the same thing as a lazy query and look at the plan Polars makes.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Colab already includes this library. This cell just makes sure it is there (it takes a few seconds).

In [ ]:
!pip install -q polars
import importlib.metadata as md
print("polars version:", md.version("polars"))

## Step 1 · Create the dataset and load it with Polars
We create the shared CSV with pandas, then read it with Polars.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 600
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n, p=[0.3, 0.2, 0.3, 0.2]),
    "quantity": rng.integers(1, 6, n),
    "price": rng.choice([199, 499, 999, 1499, 2999], n),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
})
orders["revenue"] = orders["quantity"] * orders["price"]
orders.to_csv("shop_orders.csv", index=False)
print(orders.shape, "-> saved shop_orders.csv")
orders.head()

## Step 2 · Read and inspect
`try_parse_dates` turns the date text into real dates.

In [ ]:
import polars as pl

df = pl.read_csv("shop_orders.csv", try_parse_dates=True)
print(df.shape)
print(df.schema)
df.head()

## Step 3 · Filter and select with expressions
`pl.col('name')` points at a column. Combine conditions with `&`.

In [ ]:
df.filter((pl.col("city") == "Pune") & (pl.col("revenue") > 5000)) \
  .select("order_id", "category", "revenue") \
  .sort("revenue", descending=True) \
  .head()

## Step 4 · group_by and agg
Several summaries at once, each written as an expression.

In [ ]:
df.group_by("city").agg(
    pl.len().alias("orders"),
    pl.col("revenue").sum().alias("revenue"),
    pl.col("rating").mean().round(2).alias("avg_rating"),
).sort("revenue", descending=True)

## Step 5 · New columns with with_columns
Add many columns in one call. `.dt.month()` works like in pandas.

In [ ]:
df2 = df.with_columns(
    pl.col("date").dt.month().alias("month"),
    (pl.col("revenue") * 1.18).round(0).alias("revenue_with_gst"),
    pl.when(pl.col("revenue") > 5000).then(pl.lit("large")).otherwise(pl.lit("normal")).alias("band"),
)
df2.group_by("month").agg(pl.col("revenue").sum()).sort("month")

## Step 6 · Lazy mode and the query plan
`scan_csv` does not read anything yet. `explain()` shows the optimised plan, and `collect()` runs it.

In [ ]:
query = (
    pl.scan_csv("shop_orders.csv", try_parse_dates=True)
      .filter(pl.col("category") == "Electronics")
      .group_by("city")
      .agg(pl.col("revenue").sum())
      .sort("revenue", descending=True)
)
print(query.explain())
query.collect()

## Step 7 · Speed test vs pandas
On 2 million rows the difference starts to show. Numbers vary by machine.

In [ ]:
import time
big = pd.DataFrame({"g": rng.integers(0, 1000, 2_000_000), "v": rng.random(2_000_000)})
pbig = pl.from_pandas(big)

t = time.perf_counter(); big.groupby("g")["v"].mean(); t_pd = time.perf_counter() - t
t = time.perf_counter(); pbig.group_by("g").agg(pl.col("v").mean()); t_pl = time.perf_counter() - t
print(f"pandas {t_pd:.3f}s | polars {t_pl:.3f}s")

## ✍️ Your turn
With Polars, store the **total revenue of the 'Books' category** (a number) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    expected = int(orders.loc[orders["category"] == "Books", "revenue"].sum())
    return answer == expected, f"Expected {expected}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = df.filter(pl.col("category") == "Books")["revenue"].sum()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Expression**
- **pl.col**
- **with_columns**
- **group_by / agg**
- **Lazy mode**
- **Query plan**

**Next:** 15. Dask (`dask`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/15_dask.ipynb)

## 💻 Run it on your own computer (optional)
1. Install Python 3.10+ from python.org (on Windows tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install polars
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that just opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain `.py` file, run `!` commands in the terminal instead.